# Retention Radar: from shopping data to a testable decision

**Question:** who should a retailer contact when its retention budget is limited?

This walkthrough reads saved analysis outputs. A household can occur at several historical cutoffs; it appears once in each contact roster. Risk prediction, observed shopping changes, and modeled intervention effects are different kinds of evidence.

Artifact summary generated: 2026-11-03. See the README to rebuild.

In [ ]:
import json
from pathlib import Path

import pandas as pd

outputs = Path("../data/outputs")
results = json.loads((outputs / "results.json").read_text())

## 1. Predict departures

A departure means no purchases in the next two 28-day periods. Models train on cutoffs 6–13, tune/calibrate on 16–17, and evaluate on 20–23. Household-clustered intervals account for repeated households. Overlapping intervals alone do not establish a significant difference.

In [ ]:
pd.read_parquet(outputs / "churn_model_comparison.parquet").head(15)

model,pr_auc,pr_auc_ci_lower,pr_auc_ci_upper,roc_auc,brier_score,ece,top_10_lift,top_10_lift_ci_lower,top_10_lift_ci_upper,top_10_capture,top_20_lift,top_20_capture
LightGBM,0.3803,0.3359,0.4275,0.8851,0.0561,0.0061,4.911,4.504,5.288,0.4902,3.753,0.7500
Logistic Regression,0.3197,0.2818,0.3619,0.8732,0.0586,0.0100,4.503,4.131,4.979,0.4494,3.647,0.7289
Recency Baseline,0.3063,0.2690,0.3493,0.8252,0.0599,0.0104,4.559,4.177,4.878,0.4551,3.268,0.6531
State Baseline,0.2408,0.2096,0.2754,0.7774,0.0606,0.0106,4.418,4.007,4.710,0.4410,2.916,0.5829


## 2. Inspect mistakes

At each cutoff, select the highest-risk 20%. Review missed departures and false alarms. Group counts are descriptive; small groups should not drive confident policy changes.

In [ ]:
pd.read_parquet(outputs / "churn_error_analysis.parquet").head(15)

dimension,group,observations,households,actual_departures,contacts,captured_departures,missed_departures,false_alarms,capture_rate,contact_precision
Overall,All observations,9397,2423,712,1878,534,178,1344,0.750000,0.284345
cutoff_period,20,2348,2348,171,469,127,44,342,0.742690,0.270789
cutoff_period,21,2345,2345,165,469,126,39,343,0.763636,0.268657
cutoff_period,22,2353,2353,185,470,139,46,331,0.751351,0.295745
cutoff_period,23,2351,2351,191,470,142,49,328,0.743455,0.302128
current_state,At-Risk,785,679,182,609,158,24,451,0.868132,0.259442
current_state,Champion,2027,679,6,5,1,5,4,0.166667,0.200000
current_state,Dormant,512,360,210,510,209,1,301,0.995238,0.409804
current_state,New,1,1,0,1,0,0,1,NaN,0.000000
current_state,Regular,6072,1992,314,753,166,148,587,0.528662,0.220452


## 3. Forecast customer value

Train outcomes end on day 518; test outcomes cover days 519–700. Each window lasts 26 weeks. Errors are sales dollars, not margin. A recent-spending baseline is included. Final method selection on this test is exploratory and needs another future evaluation.

In [ ]:
pd.read_parquet(outputs / "clv_comparison.parquet").head(15)

method,spearman_rank_corr,mae_dollars,rmse_dollars
Recent 26-week spend,0.8179,339.20,529.10
BG/NBD + Gamma-Gamma,0.8086,351.74,544.92
LightGBM (Tweedie),0.8112,355.99,549.55


## 4. Investigate shopping changes

Effects are differences in category spend-share changes relative to active controls. Multiply the fractional effect by 100 for percentage points. These retrospective windows include dormancy and do not validate a prospective eight-week alert.

In [ ]:
pd.read_parquet(outputs / "early_warning_categories.parquet").head(15)

commodity_desc,lapsing_hh_count,lapsing_early_spend,lapsing_late_spend,lapsing_share_change,control_share_change,net_drop_effect,rank
DIAPERS & DISPOSABLES,43,640.74,217.10,-0.03071,0.00536,-0.03608,1
CHICKEN/POULTRY,88,684.60,162.91,-0.03222,0.00108,-0.03331,2
DOG FOODS,51,530.23,137.95,-0.02735,0.00187,-0.02922,3
BEEF,343,4927.37,1772.45,-0.02261,0.00432,-0.02693,4
CHICKEN,182,1481.39,471.87,-0.02209,0.00288,-0.02498,5
INFANT FORMULA,26,839.75,247.74,-0.01042,0.01081,-0.02123,6
MEAT - MISC,148,1628.93,405.72,-0.01921,0.00080,-0.02001,7
FRZN NOVELTIES/WTR ICE,132,654.92,156.43,-0.01375,0.00268,-0.01643,8
CIGARETTES,86,1657.93,271.78,-0.01638,-0.00174,-0.01464,9
FRUIT - SHELF STABLE,170,748.45,153.82,-0.01275,-0.00010,-0.01265,10


## 5. Explore offer response

All three evaluation folds hold out entire households. Historical offers were not random. Uplift ranking remains observational, and learner selection uses these exploratory folds. Negative estimates do not establish that contacting a customer causes harm.

In [ ]:
pd.read_parquet(outputs / "uplift_eval.parquet").head(15)

learner,qini,auuc,uplift_at_10pct,uplift_at_30pct,mean_cate,fold_count,observation_count,unique_households
T-Learner,0.1598,0.6651,0.0933,0.3124,0.7249,3,28319,2466
S-Learner,0.1537,0.6310,0.0242,0.3305,0.6011,3,28319,2466
DR-Learner,0.1466,0.6533,0.0639,0.4364,0.7308,3,28319,2466
X-Learner,0.0169,0.5130,-0.1091,0.2001,0.7189,3,28319,2466


## 6. Choose the next experiment

Freeze the eligible household pool, assign one offer or control per household, and preselect the outcome and analysis. Sample-size planning uses observed spending variance and an assumed minimum worthwhile effect. If the available pool is too small, revise the study before launch.

Financial charts are scenarios using assumed response rates, margin and contact costs. No intervention experiment has been run and no campaign savings have been demonstrated.

In [ ]:
results["uplift"]["ab_test"]